## RBTC-109

In [0]:
env="p"

In [0]:
spark.sql(f"""
          ALTER TABLE `{env}-cdh`.primaryactive.collateral
          ADD COLUMN NonRealEstateCollateralStatusCode string AFTER NonRealEstateKey;
""")
spark.sql(f"""
          ALTER TABLE `{env}-cdh`.primaryactive.collateral
          ADD COLUMN NonRealEstateCollateralStatus string AFTER NonRealEstateCollateralStatusCode;
""")


In [0]:
prcollateral = spark.sql(f"""
    select distinct id, BusinessDate
    from `{env}-cdh`.landed.thesis_prcollateral
    """)

prcollateraldet = spark.sql(f"""
    select distinct collateral_id, id as iddet, collateral_status, BusinessDate as BusinessDateDet
    from `{env}-cdh`.landed.thesis_prcollateraldet
    """)

joined = prcollateral.join(
            prcollateraldet,
            (prcollateral.id == prcollateraldet.collateral_id)
            & (prcollateral.BusinessDate == prcollateraldet.BusinessDateDet),
            "left",
        )
joined.createOrReplaceTempView("joined")


spark.sql(f"""
MERGE INTO `{env}-cdh`.primaryactive.collateral a
USING joined b
ON a.CollateralKey = b.id
    AND a.NonRealEstateKey = b.iddet
    AND a.BusinessDate = b.BusinessDate
WHEN MATCHED THEN UPDATE SET a.NonRealEstateCollateralStatusCode = b.collateral_status,
                            a.NonRealEstateCollateralStatus = b.collateral_status
""")

##PMT-926

In [0]:
spark.sql(f"""
          ALTER TABLE `{env}-cdh`.primaryactive.exposure
          ADD COLUMN DueDebtAmount decimal(19,4) AFTER OutstandingTotalBalanceMLT;
""")


In [0]:
spark.sql(f'''
            ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
            ADD COLUMN DueDebtAmount decimal(19,4) AFTER IFRSGBV;                
          ''')

spark.sql(f'''
            ALTER TABLE `{env}-cdh`.derivativearchive.dexposure
            ADD COLUMN DueDebtAmount decimal(19,4) AFTER IFRSGBV;                
          ''')

In [0]:
spark.sql(f'''
MERGE INTO `{env}-cdh`.primaryactive.exposure AS target
USING (
  SELECT CASE_CODE, 
         CAEX_CUSTOM_DECIMAL_66_LCY, 
         BusinessDate
  FROM `{env}-cdh`.landed.qca_fsad_cases_extra_open
) AS source
ON target.CaseCode = source.CASE_CODE
   AND target.BusinessDate = source.BusinessDate
WHEN MATCHED AND target.sourcesystem = 'qca' THEN 
  UPDATE SET target.DueDebtAmount = source.CAEX_CUSTOM_DECIMAL_66_LCY;
''')

## RBTC-78

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE `p-cdh`.reference.ocwdata
AS SELECT * FROM  `d-cdh`.reference.extfile_ocwdata
""")

## DATA2-202

In [0]:
spark.sql(f"""
  MERGE INTO `{env}-cdh`.primaryactive.interaction i
  USING (
    SELECT
        a.cac_code AS InteractionCode
      , am.is_rpc
      , a.BusinessDate
    FROM `{env}-cdh`.landed.qca_fs_activity_leaf a
    INNER JOIN `{env}-cdh`.landed.qca_dim_activity_type c
      ON a.ACTV_ID = c.ACTV_ID
      AND a.BusinessDate = c.BusinessDate
    INNER JOIN `{env}-cdh`.landed.params_activitiesmapping_ab am
      ON CAST(c.actv_intcode AS STRING) = CAST(am.activity_code AS STRING)
      AND am.BusinessDate = c.BusinessDate
  ) src
  ON src.InteractionCode = i.InteractionCode
  AND src.BusinessDate = i.BusinessDate
  AND SourceSystem = 'qca'
  WHEN MATCHED THEN
  UPDATE SET
    i.IsRPCFlagCalc = CAST(src.is_rpc AS BOOLEAN);
""")

## DATA2-199

In [0]:
spark.sql(f"""
WITH RankedRows AS (
        SELECT 
            ROW_NUMBER() OVER (PARTITION BY BusinessDate, ExposureDerivativeID ORDER BY IngestionTimestamp) AS row_num,
            BusinessDate, ExposureDerivativeID, IngestionTimestamp
        FROM `{env}-cdh`.derivativearchive.dexposure
        WHERE BusinessDate = '2024-01-05'
)
DELETE FROM `{env}-cdh`.derivativearchive.dexposure dexp
WHERE EXISTS (
    SELECT 1
    FROM RankedRows r
    WHERE r.row_num = 1
      AND dexp.BusinessDate = r.BusinessDate
      AND dexp.ExposureDerivativeID = r.ExposureDerivativeID
      AND dexp.IngestionTimestamp = r.IngestionTimestamp
);
""")

##MINIPJS-142
### Add Zero GBV Implementation

## Steps: 
* Merge the current PR (dev/qa/prod)
* Create Table ZeroingGBVExposure in derivativeactive and view ZeroingGBVExposure in derivative
* Add new columns in DExposure
* Add ZeroingGBVExposure notebook to run after DExposure notebook in workflows
* ReRun DExposure (with ZeroingGBVExposure) from 2024-12-31 (do not run for other dates first!!)

In [0]:
spark.sql(f"""
CREATE TABLE `{env}-cdh`.derivativeactive.`zeroinggbvexposure` (
    ExposureKey STRING,
    GBVTriggerEventFlag BOOLEAN,
    GBVTriggerEventDate DATE,
    GBVTriggerEventType STRING,
    GBVCappedAmount DOUBLE,
    GBVCappedDate DATE,
    GBVTailExposure DOUBLE,
    GBVFunctionResult DOUBLE,
    BusinessDate DATE
);
""")

In [0]:
spark.sql(f"""
CREATE VIEW `{env}-cdh`.derivative.zeroinggbvexposure AS
SELECT 
    ExposureKey,
    GBVTriggerEventFlag,
    GBVTriggerEventDate,
    GBVTriggerEventType,
    GBVCappedAmount,
    GBVCappedDate,
    GBVTailExposure,
    GBVFunctionResult,
    BusinessDate 
FROM `{env}-cdh`.derivativeactive.zeroinggbvexposure;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD COLUMN SubProductDescription STRING AFTER ProductCode;

ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD COLUMN GBVTriggerEventFlag BOOLEAN AFTER GBV;

ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD COLUMN GBVTriggerEventDate date AFTER GBVTriggerEventFlag ;

ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD COLUMN GBVTriggerEventType string AFTER GBVTriggerEventDate ;

ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD COLUMN TailExposure double AFTER GBVTriggerEventType ;

ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD COLUMN GBVCappedAmount double AFTER TailExposure ;

ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD COLUMN GBVCappedDate date AFTER GBVCappedAmount ;

ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD COLUMN GBVFunctionResult double AFTER GBVCappedDate ;
""")

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN SubProductDescription STRING AFTER ProductCode;

ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN GBVTriggerEventFlag BOOLEAN AFTER GBV;

ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN GBVTriggerEventDate date AFTER GBVTriggerEventFlag ;

ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN GBVTriggerEventType string AFTER GBVTriggerEventDate ;

ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN TailExposure double AFTER GBVTriggerEventType ;

ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN GBVCappedAmount double AFTER TailExposure ;

ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN GBVCappedDate date AFTER GBVCappedAmount ;

ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN GBVFunctionResult double AFTER GBVCappedDate ;

## DATA2-195

In [0]:
# add column IsActiveFlagInt to dexposure
spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD COLUMN IsActiveFlagInt SMALLINT after IsActiveFlag;
""")

spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativearchive.dexposure
ADD COLUMN IsActiveFlagInt SMALLINT after IsActiveFlag;
""")

spark.sql( f"""
UPDATE `{env}-cdh`.derivativeactive.dexposure
SET IsActiveFlagInt = CAST(IsActiveFlag AS SMALLINT);
""")

spark.sql( f"""
UPDATE `{env}-cdh`.derivativearchive.dexposure
SET IsActiveFlagInt = CAST(IsActiveFlag AS SMALLINT);
""")

In [0]:
# Enforce a constraint to ensure the relationship between IsActiveFlag and IsActiveFlagInt is maintained
spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativearchive.dexposure
ADD CONSTRAINT valid_is_active_flag CHECK (IsActiveFlagInt = CAST(IsActiveFlag AS SMALLINT));
""")

spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativeactive.dexposure
ADD CONSTRAINT valid_is_active_flag CHECK (IsActiveFlagInt = CAST(IsActiveFlag AS SMALLINT));
""")


In [0]:
# Rename derivativearchive.DExposure & derivativeactive.DExposure for backup
spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativearchive.DExposure RENAME TO `{env}-cdh`.derivativearchive.DExposure_backup;
""")

spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativeactive.DExposure RENAME TO `{env}-cdh`.derivativeactive.DExposure_backup;
""")

In [0]:
# Create derivativearchive.DExposure with clustering by (BusinessDate, ExposureKey, IsActiveFlagInt)
spark.sql(f'''
CREATE OR REPLACE TABLE `{env}-cdh`.derivativearchive.DExposure
CLUSTER BY (BusinessDate, ExposureKey, IsActiveFlagInt)
TBLPROPERTIES (
            'delta.logRetentionDuration' = 'interval 5 weeks',
            'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
            'delta.minReaderVersion' = '3',
            'delta.minWriterVersion' = '7',
            'delta.columnMapping.mode' = 'name',
            'delta.dataSkippingStatsColumns' = 'BusinessDate, ExposureKey,IsActiveFlagInt'
            ) 
  as select * from   `{env}-cdh`.derivativearchive.DExposure_backup 
''')

# Create derivativeactive.DExposure with clustering by (BusinessDate, ExposureKey, IsActiveFlagInt)
spark.sql(f'''
CREATE OR REPLACE TABLE `{env}-cdh`.derivativeactive.DExposure
CLUSTER BY (BusinessDate, ExposureKey, IsActiveFlagInt)
TBLPROPERTIES (
            'delta.logRetentionDuration' = 'interval 5 weeks',
            'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
            'delta.minReaderVersion' = '3',
            'delta.minWriterVersion' = '7',
            'delta.columnMapping.mode' = 'name',
            'delta.dataSkippingStatsColumns' = 'BusinessDate, ExposureKey,IsActiveFlagInt'
            ) 
  as select * from   `{env}-cdh`.derivativeactive.DExposure_backup 
''')

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



In [0]:
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `{env}-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')